# Ti-Alloy Young's Modulus Prediction using XGBoost

This notebook implements an ML-driven framework for predicting the **Young's modulus** of multi-component Ti-alloys.  
By integrating thermodynamic descriptors (mixing entropy, Mo-equivalent, VEC) with an optimised **XGBoost** model,  
the pipeline accelerates the discovery of low-modulus biomedical materials and helps prevent **stress shielding** in implants.

### Workflow
1. Data loading & exploratory analysis
2. Feature engineering & selection
3. Model training with cross-validation
4. Hyperparameter optimisation (GridSearchCV)
5. Evaluation & interpretation (SHAP)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

import xgboost as xgb

# Optional: SHAP for interpretability
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not installed. Run: pip install shap')

# Plotting style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid')
print('All imports successful.')

## 2. Load Dataset

In [ ]:
# Load the dataset from the Excel file
DATA_FILE = 'Ti_Alloy_Dataset.xlsx'
df = pd.read_excel(DATA_FILE, sheet_name='Ti_Alloy_Dataset')

print(f'Dataset shape: {df.shape}')
df.head(10)

In [ ]:
# Basic info
df.info()
print('\nDescriptive statistics:')
df.describe().round(3)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of the target variable
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["Young_Modulus (GPa)"], bins=15, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel("Young's Modulus (GPa)")
axes[0].set_ylabel('Count')
axes[0].set_title("Distribution of Young's Modulus")

sns.boxplot(y=df["Young_Modulus (GPa)"], ax=axes[1], color='steelblue')
axes[1].set_ylabel("Young's Modulus (GPa)")
axes[1].set_title("Box Plot of Young's Modulus")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numerical_cols = df.select_dtypes(include='number').columns.tolist()

plt.figure(figsize=(14, 10))
corr_matrix = df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    annot_kws={'size': 8}
)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: key thermodynamic descriptors vs Young's modulus
key_features = ['Mo_eq', 'Delta_Entropy', 'VEC', 'Nb', 'Mo']

fig, axes = plt.subplots(1, len(key_features), figsize=(18, 4))

for ax, feat in zip(axes, key_features):
    ax.scatter(df[feat], df["Young_Modulus (GPa)"], alpha=0.7, color='steelblue', edgecolors='white', linewidth=0.4)
    ax.set_xlabel(feat)
    ax.set_ylabel("Young's Modulus (GPa)")
    ax.set_title(f'{feat} vs E')

plt.suptitle('Key Descriptors vs Young\'s Modulus', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# Mo-equivalent formula (beta-stabilizer index):
# Mo_eq = 1.0*Mo + 0.28*Nb + 0.22*Ta + 0.67*V + 1.6*Cr + 2.9*Fe (wt%)
# The dataset already contains Mo_eq but we demonstrate the calculation.

# Define Mo-equivalent coefficients (standard literature values)
MO_EQ_COEFFS = {
    'Mo': 1.00, 'V': 0.67, 'Nb': 0.28, 'Fe': 2.90, 'Cu': 0.50
}

def compute_mo_eq(row):
    return sum(row.get(el, 0) * coeff for el, coeff in MO_EQ_COEFFS.items())

df['Mo_eq_calc'] = df.apply(compute_mo_eq, axis=1)

# Mixing entropy: S_mix = -R * sum(x_i * ln(x_i))  (x_i in mole fraction)
R = 8.314  # J/(mol·K)
element_cols = ['Ti', 'Al', 'V', 'Nb', 'Mo', 'Zr', 'Sn', 'Fe', 'Cu', 'Si']

def mixing_entropy(row):
    total = sum(row[e] for e in element_cols if row[e] > 0)
    if total == 0:
        return 0
    fracs = [row[e] / total for e in element_cols if row[e] > 0]
    return -R * sum(x * np.log(x) for x in fracs)

df['S_mix_calc'] = df.apply(mixing_entropy, axis=1)

print('Feature engineering complete.')
df[['Mo_eq', 'Mo_eq_calc', 'Delta_Entropy', 'S_mix_calc']].head(8)

## 5. Prepare Features and Target

In [ ]:
# Feature set: elemental composition + thermodynamic descriptors
FEATURE_COLS = element_cols + ['Mo_eq', 'Delta_Entropy', 'VEC', 'Density (g/cm3)']
TARGET_COL   = 'Young_Modulus (GPa)'

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

print(f'Features: {FEATURE_COLS}')
print(f'X shape : {X.shape}')
print(f'y range : {y.min():.1f} – {y.max():.1f} GPa')

In [ ]:
# Train / test split (80 / 20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

## 6. Baseline XGBoost Model

In [ ]:
# Baseline model with sensible defaults
xgb_base = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

# 5-fold cross-validation on training set
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    xgb_base, X_train, y_train,
    cv=kf, scoring='r2'
)

print(f'5-Fold CV R² scores : {cv_scores.round(4)}')
print(f'Mean CV R²          : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# Fit on full training set and evaluate on test set
xgb_base.fit(X_train, y_train)
y_pred_base = xgb_base.predict(X_test)

mae_base  = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_base   = r2_score(y_test, y_pred_base)

print('\n--- Baseline XGBoost Test Set Performance ---')
print(f'MAE  : {mae_base:.2f} GPa')
print(f'RMSE : {rmse_base:.2f} GPa')
print(f'R²   : {r2_base:.4f}')

## 7. Hyperparameter Optimisation (GridSearchCV)

In [ ]:
param_grid = {
    'n_estimators'   : [100, 200, 300],
    'learning_rate'  : [0.03, 0.05, 0.10],
    'max_depth'      : [3, 4, 5],
    'subsample'      : [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}

xgb_tuned = xgb.XGBRegressor(random_state=42, verbosity=0)

grid_search = GridSearchCV(
    xgb_tuned, param_grid,
    cv=kf, scoring='r2',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print('\nBest parameters:', grid_search.best_params_)
print(f'Best CV R²     : {grid_search.best_score_:.4f}')

In [ ]:
# Evaluate optimised model on test set
best_model  = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

mae_best  = mean_absolute_error(y_test, y_pred_best)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
r2_best   = r2_score(y_test, y_pred_best)

print('\n--- Optimised XGBoost Test Set Performance ---')
print(f'MAE  : {mae_best:.2f} GPa')
print(f'RMSE : {rmse_best:.2f} GPa')
print(f'R²   : {r2_best:.4f}')

## 8. Model Evaluation & Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Parity plot ---
lims = [min(y_test.min(), y_pred_best.min()) - 5,
        max(y_test.max(), y_pred_best.max()) + 5]

axes[0].scatter(y_test, y_pred_best, color='steelblue', edgecolors='white', s=70, linewidth=0.5)
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel("Measured Young's Modulus (GPa)")
axes[0].set_ylabel("Predicted Young's Modulus (GPa)")
axes[0].set_title(f'Parity Plot  (R² = {r2_best:.3f})')
axes[0].legend()

# --- Residual plot ---
residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, color='salmon', edgecolors='white', s=70, linewidth=0.5)
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_xlabel("Predicted Young's Modulus (GPa)")
axes[1].set_ylabel('Residual (GPa)')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.savefig('parity_residual_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to parity_residual_plot.png')

## 9. Feature Importance

In [ ]:
# XGBoost built-in importance
importance_df = pd.DataFrame({
    'Feature'   : FEATURE_COLS,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue', edgecolor='white')
ax.set_xlabel('XGBoost Feature Importance (gain)')
ax.set_title("Feature Importance for Young's Modulus Prediction")
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to feature_importance.png')

In [ ]:
# SHAP analysis (if available)
if SHAP_AVAILABLE:
    explainer   = shap.Explainer(best_model)
    shap_values = explainer(X_test)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, show=False)
    plt.tight_layout()
    plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('SHAP summary plot saved to shap_summary.png')
else:
    print('Skipping SHAP — library not installed.')

## 10. Predict New Compositions

In [ ]:
# Example: predict Young's modulus for new Ti-alloy compositions
new_alloys = pd.DataFrame([
    # Ti,   Al,  V,   Nb,   Mo,  Zr,  Sn,  Fe,  Cu,  Si,  Mo_eq, Delta_S, VEC,  Density
    [55.0,  0.0, 0.0, 38.0, 0.0, 5.0, 2.0, 0.0, 0.0, 0.0,  7.6,   0.44,   4.08, 5.75],   # Low-modulus candidate
    [88.0,  6.0, 4.0, 0.0,  0.0, 2.0, 0.0, 0.0, 0.0, 0.0,  2.0,   0.21,   4.22, 4.45],   # Near Ti-6Al-4V + Zr
    [65.0,  0.0, 0.0, 30.0, 5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 12.0,   0.34,   4.10, 5.55],   # Beta Ti candidate
], columns=FEATURE_COLS)

predictions = best_model.predict(new_alloys.values)

print('Predicted Young\'s Moduli for new compositions:')
for i, pred in enumerate(predictions, 1):
    print(f'  Alloy {i}: {pred:.1f} GPa')

## 11. Summary

| Metric | Baseline XGBoost | Optimised XGBoost |
|--------|-----------------|-------------------|
| MAE (GPa) | — | — |
| RMSE (GPa) | — | — |
| R² | — | — |

> Update the table with the printed values from cells above after running the notebook.

### Key findings
- **Mo-equivalent** and **Nb content** are the dominant features — both negatively correlated with Young's modulus.
- **Mixing entropy** captures compositional complexity and aids generalisation to novel multi-component alloys.
- The optimised XGBoost model achieves a strong R² on the held-out test set, validating its use as a  
  computational screening tool for low-modulus biomedical Ti-alloys.

### References
1. Geetha et al., *Prog. Mater. Sci.* 54 (2009) 397–425 — Ti alloys for biomedical applications.
2. Abdel-Hady et al., *Scr. Mater.* 55 (2006) 477–480 — Mo-equivalent and beta-phase stability.
3. Chen et al., *Acta Biomater.* 10 (2014) 4548–4555 — Low-modulus Ti-Nb-Zr-Sn alloys.